# GoEmotions Fine-Tuning Notebook

This notebook fine-tunes a multilingual BERT sentiment model on the Hugging Face **GoEmotions** dataset.

We are doing this because the current project already has a replaceable model layer, and this training step will later plug into:

- `nlp/trained_bert_model.py`
- `nlp/sentiment_service.py`

## Colab note

This notebook is written so it can run locally or in Google Colab.

- In Colab, the trained model can be uploaded directly to Google Drive.
- Locally, you can still save the model folder beside the repository if you want.
- The first run may need internet access to download the dataset and base model from Hugging Face.

## What this notebook does

1. Loads GoEmotions from Hugging Face.
2. Converts the original multi-label data into a simpler single-label setup by keeping the first label as `primary_emotion`.
3. Fine-tunes `nlptown/bert-base-multilingual-uncased-sentiment` for emotion classification.
4. Evaluates the model with accuracy, precision, recall, F1, a classification report, and a confusion matrix.
5. Saves the trained model, tokenizer, and metadata for later integration with the Big Data pipeline.


## 2. Install and Import Dependencies

If you are running this notebook in a fresh environment, the commented install line below is the safe first step.
The imports that follow are the exact libraries needed for this training notebook.


In [ ]:
# If you are running this notebook in a fresh environment, uncomment the next line.
# !pip install -q datasets transformers torch evaluate scikit-learn pandas numpy matplotlib
# If evaluate is missing, the notebook still works because the real metrics use sklearn.

from __future__ import annotations

import json
import os
import warnings
from datetime import datetime, timezone
from inspect import signature
from pathlib import Path

try:
    import evaluate
except ModuleNotFoundError:
    # Optional dependency: the notebook can run without evaluate because sklearn handles the metrics.
    evaluate = None
    print("Optional package 'evaluate' is not installed. The notebook will continue with sklearn metrics only.")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)
print("Dependencies imported successfully.")

## 3. Configuration

We define the base model, dataset, and training hyperparameters here.

We also detect the runtime environment and the device:

- `cuda` if a GPU is available
- otherwise `cpu`

This matters because BERT fine-tuning is much faster on GPU.


In [ ]:
# Core training configuration.
# These values are chosen so the notebook stays readable and easy to modify later.
BASE_MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
DATASET_NAME = "mrm8488/goemotions"
OUTPUT_DIR = Path("fine_tuned_bert")
MAX_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
TEST_SIZE = 0.2
RANDOM_SEED = 42

# The notebook saves the trained model into a normal folder in the current runtime.
# In Colab, we later copy that folder to Google Drive.
OUTPUT_DIR_PATH = OUTPUT_DIR.resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)

print(f"Device detected: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR_PATH}")
print(f"Base model: {BASE_MODEL_NAME}")
print(f"Dataset: {DATASET_NAME}")


## 4. Load Dataset

Here we download the GoEmotions dataset from Hugging Face and inspect its structure.

Expected output:

- dataset splits
- feature schema
- example rows
- label information

If you are working offline, you will need internet access or a local copy of the dataset.


In [ ]:
# Load the dataset directly from Hugging Face.
# The first download may take a little time on a fresh machine.
dataset = load_dataset(DATASET_NAME)

print("Dataset loaded successfully.")
print(dataset)
print("\nAvailable splits:", list(dataset.keys()))

for split_name in dataset.keys():
    split = dataset[split_name]
    print(f"\n--- {split_name.upper()} SPLIT ---")
    print(f"Number of rows: {len(split)}")
    print("Features:")
    print(split.features)
    print("Example rows:")
    display(split.select(range(min(3, len(split)))).to_pandas())

# This GoEmotions revision stores one text column plus one binary column per emotion.
# We keep the text column separate and derive labels from the emotion columns later.
reference_split = dataset[next(iter(dataset.keys()))]
feature_names = list(reference_split.features.keys())
text_candidates = ["text", "sentence", "content", "comment"]
TEXT_COLUMN = next((col for col in text_candidates if col in feature_names), None)

if TEXT_COLUMN is None:
    raise ValueError(f"Could not detect a text column from features: {feature_names}")

print(f"\nDetected text column: {TEXT_COLUMN}")
print("The emotion columns will be derived from the dataset schema in the next step.")


## 5. Prepare Labels

GoEmotions is originally a **multi-label** dataset, which means one text can have multiple emotion labels.

For this first training version we simplify it to a **single-label** problem by keeping only the first label in each example.

Why do this?

- It is easier to train and explain.
- It keeps the notebook manageable for a first version.
- Later, we can upgrade to true multi-label emotion classification without changing the whole project.

The new label column will be called `primary_emotion`.

In [ ]:
# Extract the authoritative GoEmotions label names from the dataset schema.
# This revision exposes one binary column per emotion, so we use those column names as the label set.
# The ordering here matters because we will use it to pick the first active emotion as the primary label.
EXPECTED_GOEMOTIONS_LABELS = [
    "admiration",
    "amusement",
    "anger",
    "annoyance",
    "approval",
    "caring",
    "confusion",
    "curiosity",
    "desire",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "excitement",
    "fear",
    "gratitude",
    "grief",
    "joy",
    "love",
    "nervousness",
    "optimism",
    "pride",
    "realization",
    "relief",
    "remorse",
    "sadness",
    "surprise",
    "neutral",
]

# Only keep the emotion columns that actually exist in this dataset revision.
EMOTION_COLUMNS = [label for label in EXPECTED_GOEMOTIONS_LABELS if label in feature_names]
missing_expected = [label for label in EXPECTED_GOEMOTIONS_LABELS if label not in EMOTION_COLUMNS]

if missing_expected:
    print("Some expected GoEmotions labels were not found in the dataset schema:")
    print(missing_expected)

label_names = EMOTION_COLUMNS
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}
num_labels = len(label_names)

print(f"Number of GoEmotions labels: {num_labels}")
print("Label names:")
print(label_names)


def extract_primary_label(row):
    # The dataset is multi-label, but this first version trains a single-label classifier.
    # We choose the first active emotion in the label order defined above.
    if row is None:
        return None

    # Rows marked as very unclear are harder to learn from, so we skip them.
    if bool(row.get("example_very_unclear", False)):
        return None

    active_labels = [label for label in label_names if int(row.get(label, 0)) == 1]
    if not active_labels:
        return None

    return label2id[active_labels[0]]


def prepare_split_as_dataframe(hf_split):
    # Convert a Hugging Face split into a clean pandas DataFrame.
    # We keep only the primary emotion so the first training pass behaves like a standard single-label classifier.
    rows = []
    for row in hf_split:
        text = row.get(TEXT_COLUMN)
        primary_id = extract_primary_label(row)

        if primary_id is None:
            continue
        if text is None or str(text).strip() == "":
            continue
        if primary_id not in id2label:
            continue

        rows.append(
            {
                "text": str(text),
                "labels": int(primary_id),
                "primary_emotion": id2label[int(primary_id)],
            }
        )

    return pd.DataFrame(rows)


split_dfs = {}
all_split_names = set(dataset.keys())

if {"train", "validation", "test"}.issubset(all_split_names):
    split_dfs["train"] = prepare_split_as_dataframe(dataset["train"])
    split_dfs["validation"] = prepare_split_as_dataframe(dataset["validation"])
    split_dfs["test"] = prepare_split_as_dataframe(dataset["test"])
    print("Using the dataset's existing train/validation/test splits.")
elif {"train", "test"}.issubset(all_split_names):
    # If validation is missing, create it from the train split.
    train_full_df = prepare_split_as_dataframe(dataset["train"])
    test_df = prepare_split_as_dataframe(dataset["test"])

    try:
        train_df, validation_df = train_test_split(
            train_full_df,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=train_full_df["labels"],
        )
    except ValueError:
        # If any class is too small for stratification, fall back to a standard random split.
        train_df, validation_df = train_test_split(
            train_full_df,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=None,
        )

    split_dfs["train"] = train_df.reset_index(drop=True)
    split_dfs["validation"] = validation_df.reset_index(drop=True)
    split_dfs["test"] = test_df.reset_index(drop=True)
    print("Validation split created from the training data.")
else:
    # Generic fallback for any dataset variant that exposes only one split.
    all_frames = [prepare_split_as_dataframe(dataset[name]) for name in dataset.keys()]
    combined_df = pd.concat(all_frames, ignore_index=True)

    try:
        train_df, temp_df = train_test_split(
            combined_df,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=combined_df["labels"],
        )
        validation_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=RANDOM_SEED,
            stratify=temp_df["labels"],
        )
    except ValueError:
        train_df, temp_df = train_test_split(
            combined_df,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=None,
        )
        validation_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=RANDOM_SEED,
            stratify=None,
        )

    split_dfs["train"] = train_df.reset_index(drop=True)
    split_dfs["validation"] = validation_df.reset_index(drop=True)
    split_dfs["test"] = test_df.reset_index(drop=True)
    print("Dataset was manually split into train/validation/test.")

print("\nPrepared split sizes:")
for split_name, df in split_dfs.items():
    print(f"- {split_name}: {len(df)} examples")

all_processed_df = pd.concat(split_dfs.values(), ignore_index=True)
print("\nTop label counts across the processed dataset:")
label_distribution = all_processed_df["primary_emotion"].value_counts().reset_index()
label_distribution.columns = ["primary_emotion", "count"]
display(label_distribution)

plt.figure(figsize=(14, 6))
label_distribution.head(15).plot(kind="bar", x="primary_emotion", y="count", legend=False)
plt.title("Top GoEmotions labels in the processed dataset")
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\nPrimary emotion conversion completed successfully.")
print("Note: the dataset is still naturally multi-label; we are simplifying it to one label per example for this first version.")


## 6. Preprocess Dataset

This step converts the cleaned data into tokenized training examples.

What tokenization means:

- The tokenizer turns text into numeric tokens that BERT can understand.
- We use `padding="max_length"` so all sequences have the same length.
- We use `truncation=True` so long examples are cut to `MAX_LENGTH` tokens.

Expected output:

- train/validation/test datasets ready for the Trainer
- `input_ids`, `attention_mask`, and `labels` columns

In [ ]:
# Convert each pandas split into a Hugging Face Dataset.
# Then tokenize the text so the model receives numeric inputs instead of raw strings.
raw_datasets = DatasetDict(
    {
        split_name: Dataset.from_pandas(df, preserve_index=False)
        for split_name, df in split_dfs.items()
    }
)

print(raw_datasets)

# The tokenizer converts text into input_ids and attention_mask.
# We keep the labels column untouched so the Trainer can learn the emotion class.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_datasets = raw_datasets.map(tokenize_batch, batched=True)

# Keep only the columns the model really needs.
# The Trainer can still use labels while the original text stays available in raw_datasets for inspection.
columns_to_keep = ["input_ids", "attention_mask", "labels"]

for split_name in tokenized_datasets.keys():
    current_columns = tokenized_datasets[split_name].column_names
    columns_to_remove = [col for col in current_columns if col not in columns_to_keep]
    tokenized_datasets[split_name] = tokenized_datasets[split_name].remove_columns(columns_to_remove)

print("\nTokenized dataset example:")
print(tokenized_datasets["train"][0])
print("\nTokenized split sizes:")
for split_name, split_dataset in tokenized_datasets.items():
    print(f"- {split_name}: {len(split_dataset)} examples")


## 7. Load Base Model

We start from `nlptown/bert-base-multilingual-uncased-sentiment` and replace its classification head so it matches the number of GoEmotions labels.

Why `ignore_mismatched_sizes=True`?

- The original model was trained for sentiment-style classification.
- GoEmotions has a different label space.
- The classification head weights do not match, so we allow Hugging Face to reinitialize that head safely.

The encoder weights are reused, but the final classifier is adapted to the new task.

In [ ]:
# Load the base model and adapt it to the GoEmotions label space.
# We reuse the multilingual BERT encoder and replace the classification head.
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# Make the task type explicit for clarity.
model.config.problem_type = "single_label_classification"
model.to(DEVICE)

print("Model loaded successfully.")
print(f"Model device: {DEVICE}")
print(f"Number of output labels: {model.config.num_labels}")
print("Model head adapted for GoEmotions.")

## 8. Metrics

We evaluate the model with standard classification metrics:

- **Accuracy**: overall correctness
- **Precision**: how many predicted labels were correct
- **Recall**: how many real labels were found
- **F1-score**: balance between precision and recall

We use the weighted average because GoEmotions is not perfectly balanced across labels.

In [ ]:
# We use sklearn metrics because they are easy to read and give a full report.
# The evaluate library is optional here; sklearn gives us the clearest training notebook output.

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


print("Metric helper ready.")
print("We will mainly rely on sklearn for the training report and classification breakdown.")

## 9. Training Setup

We configure the Hugging Face `Trainer` here.

Important settings:

- `load_best_model_at_end=True` keeps the checkpoint with the best F1 score.
- `metric_for_best_model="f1"` tells the Trainer what to optimize.
- We set the evaluation/save strategy in a way that works with both older and newer Transformers versions.

CPU warning:

- Training can be slow on CPU.
- GPU is strongly recommended if available.


In [ ]:
# Hugging Face Trainer configuration.
# We use introspection so the notebook works with both older and newer Transformers releases.
training_args_kwargs = {
    "output_dir": str(OUTPUT_DIR_PATH),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "report_to": "none",
    "fp16": torch.cuda.is_available(),
}

training_args_params = signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_args_params:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args_kwargs["save_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Trainer signatures changed across Transformers versions.
# Older versions accept tokenizer=...; newer versions prefer processing_class=....
trainer_params = signature(Trainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer created successfully.")
print("Training will save the best checkpoint according to F1 score.")


## 10. Train Model

This is the main fine-tuning step.

The notebook intentionally prints progress so you can verify the training has started.

In [ ]:
print("Starting training now. If you are on CPU, this can take a while.")
print("If GPU is enabled, training should be much faster.")

train_result = trainer.train()
print("Training finished.")
print(train_result)


## 11. Evaluate Model

After training, we evaluate the model on the validation and test sets.

We also display:

- the classification report
- the confusion matrix

The confusion matrix helps us see which emotions the model confuses most often.

In [ ]:
# Evaluate on validation and test splits.
validation_metrics = trainer.evaluate(tokenized_datasets["validation"])
test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("\nValidation metrics:")
print(validation_metrics)
print("\nTest metrics:")
print(test_metrics)

# Build a more detailed report from test predictions.
prediction_output = trainer.predict(tokenized_datasets["test"])
test_logits = prediction_output.predictions
y_true = prediction_output.label_ids
y_pred = np.argmax(test_logits, axis=-1)

print("\nClassification report on the test set:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_names,
        zero_division=0,
        digits=4,
    )
)

# Confusion matrices can become large for 28 labels, but they are still useful for debugging.
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("GoEmotions confusion matrix on the test split")
plt.tight_layout()
plt.show()

print("Evaluation completed successfully.")


## 12. Test Predictions

This section creates a small helper function that runs a single text through the trained model.

What softmax does:

- It converts raw model scores (logits) into probabilities that sum to 1.
- The largest probability becomes the `confidence` score for the predicted emotion.

We also map the predicted emotion back to a broader sentiment bucket for compatibility with the current Big Data pipeline.

In [ ]:
# Emotion-to-sentiment mapping keeps the future project integration simple.
# The current pipeline expects sentiment-like output, so we map emotions into positive/negative/neutral buckets.
POSITIVE_EMOTIONS = {
    "admiration",
    "amusement",
    "approval",
    "caring",
    "desire",
    "excitement",
    "gratitude",
    "joy",
    "love",
    "optimism",
    "pride",
    "relief",
}

NEGATIVE_EMOTIONS = {
    "anger",
    "annoyance",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "fear",
    "grief",
    "nervousness",
    "remorse",
    "sadness",
}

NEUTRAL_EMOTIONS = {
    "neutral",
    "confusion",
    "curiosity",
    "realization",
    "surprise",
}


def emotion_to_sentiment(emotion: str) -> str:
    emotion = str(emotion or "").strip().lower()
    if emotion in POSITIVE_EMOTIONS:
        return "positive"
    if emotion in NEGATIVE_EMOTIONS:
        return "negative"
    if emotion in NEUTRAL_EMOTIONS:
        return "neutral"
    return "neutral"


def predict_emotion(text: str, top_k: int = 5) -> dict:
    """Predict the primary emotion and convert it into the pipeline-compatible sentiment format."""

    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0]
        probabilities = torch.softmax(logits, dim=-1).detach().cpu().numpy()

    ranked_indices = np.argsort(probabilities)[::-1][:top_k]
    emotion_scores = [
        {"label": id2label[int(idx)], "score": float(probabilities[int(idx)])}
        for idx in ranked_indices
    ]

    primary_emotion = emotion_scores[0]["label"]
    confidence = emotion_scores[0]["score"]
    sentiment = emotion_to_sentiment(primary_emotion)

    return {
        "sentiment": sentiment,
        "confidence": confidence,
        "primary_emotion": primary_emotion,
        "emotion_scores": emotion_scores,
        "model_name": "fine_tuned_goemotions_bert",
        "model_version": "v2",
    }


sample_texts = [
    "I love this match, Morocco played very well!",
    "This is terrible and disappointing.",
    "I am scared about what will happen next.",
    "This video is very informative.",
    "I feel sad and angry.",
]

for text in sample_texts:
    result = predict_emotion(text)
    print("\nTEXT:", text)
    print("PREDICTION:", result)


## 13. Save Model

We save everything needed for later integration:

- model weights
- tokenizer
- metadata JSON

This makes the trained artifact easy to load later from `nlp/trained_bert_model.py`.

In [ ]:
# Save the trained model, tokenizer, and metadata so the project can load them later.
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

model.save_pretrained(OUTPUT_DIR_PATH)
tokenizer.save_pretrained(OUTPUT_DIR_PATH)

metadata = {
    "model_name": "fine_tuned_goemotions_bert",
    "base_model": BASE_MODEL_NAME,
    "dataset": DATASET_NAME,
    "task": "emotion_multiclass_primary_label",
    "model_version": "v2",
    "labels": label_names,
    "num_labels": num_labels,
    "max_length": MAX_LENGTH,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "output_format": {
        "sentiment": "mapped from primary_emotion",
        "confidence": "max softmax probability",
        "primary_emotion": "predicted GoEmotions label",
        "emotion_scores": "top emotion probabilities",
    },
}

metadata_path = OUTPUT_DIR_PATH / "model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Model saved to: {OUTPUT_DIR_PATH}")
print(f"Tokenizer saved to: {OUTPUT_DIR_PATH}")
print(f"Metadata saved to: {metadata_path}")
print("The saved model can later be loaded from nlp/trained_bert_model.py.")

## 13. Upload Model to Google Drive

This section is Colab-first. It mounts Google Drive and copies the final model files into a Drive folder so you can reuse them later.

It uploads only the final inference files, not the training checkpoints.


In [ ]:
import shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Copy only the final model files, not the training checkpoints.
source_dir = OUTPUT_DIR_PATH
drive_dir = Path('/content/drive/MyDrive/fine_tuned_bert')
drive_dir.mkdir(parents=True, exist_ok=True)

for item in source_dir.iterdir():
    if item.is_file():
        shutil.copy2(item, drive_dir / item.name)

print(f'Uploaded model files to: {drive_dir}')
print('Files uploaded:')
for item in sorted(drive_dir.iterdir()):
    if item.is_file():
        print('-', item.name)


## 14. Emotion to Sentiment Mapping

Later in the project, the dashboard and streaming pipeline still expect a sentiment-like output.

This helper converts emotions into the broader groups:

- positive
- negative
- neutral

If an emotion is unknown, we default to `neutral` so the pipeline stays stable.

## 15. Future Integration Notes

After training, the project can switch to the fine-tuned model by changing the model configuration to something like:

- `MODEL_PROVIDER=trained`
- `MODEL_NAME=fine_tuned_goemotions_bert`
- `MODEL_VERSION=v2`

Then `nlp/trained_bert_model.py` can load the saved files from `models/fine_tuned_bert/`.

The output will stay compatible with the current project shape:

```json
{
  "sentiment": "positive",
  "confidence": 0.91,
  "primary_emotion": "joy",
  "emotion_scores": [
    {"label": "joy", "score": 0.91},
    {"label": "optimism", "score": 0.62}
  ],
  "model_name": "fine_tuned_goemotions_bert",
  "model_version": "v2"
}
```

# Final Summary

This notebook fine-tunes `nlptown/bert-base-multilingual-uncased-sentiment` on the Hugging Face GoEmotions dataset.

What was trained:

- a multilingual BERT sequence classifier
- adapted from a sentiment head to a 28-class GoEmotions label space
- simplified to single-label classification using the first label as `primary_emotion`

Where the model is saved:

- in the current runtime folder: `fine_tuned_bert/`
- in Colab: copy the saved files to Google Drive with the upload cell

How it will be used later:

- `nlp/trained_bert_model.py` will load the saved model
- `nlp/sentiment_service.py` will expose it through the same stable interface used by the current pipeline
- the streaming job and dashboard can keep the same output contract while using a better fine-tuned model

Current limitation:

- this first version is single-label only, even though GoEmotions is naturally multi-label
- CPU training can be slow
- internet access may be needed for the first download
